# Master pipeline — methylation × RNA in Arabidopsis leaf

Single notebook collecting the paper-grade analyses, factored on top of `dmw_utils.py`.
Each section corresponds to one figure or table in the paper.

| Section | Stage | Figure / table |
|---|---|---|
| 0 | Setup | — |
| 1 | Data loading (RNA + gene MCDS + DMW MCDS) | — |
| 2 | Per-cluster methylation residuals | — |
| 3 | Gene-level WLS + Top-300 heatmap | **Fig 1** |
| 4 | DMW-level WLS + Top-300 heatmap | **Fig 2** |
| 5 | Cluster-specific hypomethylation catalogs (gene + DMW) | **Table 1** |
| 6 | col / rdd / met side-by-side residual panels | **Fig 3** |
| 7 | col-anchored bulk-shift comparison | **Fig 3 inset** |
| 8 | (Supplement) 9-condition subset heatmaps | **Supp Fig S1** |

Anything not in the table above belongs in a separate supplementary notebook.

## 0. Setup

In [ ]:
import os, sys
sys.path.insert(0, '/ceph/MethDev/pbio/kay')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import dmw_utils as du
from dmw_utils import CLUSTER_MAPPING, CLUSTER_ORDER, GENOTYPES, PATHS

FIG_DIR  = PATHS['fig_outdir']
DATA_DIR = PATHS['data_outdir']
os.makedirs(FIG_DIR,  exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

## 1. Data loading

Three datasets:
- **RNA** counts (genes × cells) collapsed to log1p-CPM means per cluster.
- **Gene MCDS** (gene-body methylation; col-0 only).
- **DMW MCDS** (all three genotypes; curated to the 3,124-DMW set from chunks_CG_minfilt.gff).

In [ ]:
cluster_rna, meta = du.load_rna_cluster_matrix()
print(f'RNA cluster matrix: {cluster_rna.shape}  (genes × clusters)')
print(f'Meta cells: {len(meta)}')

In [ ]:
gene_mcds = du.load_gene_mcds()
dmw_mcds  = du.load_dmw_mcds(curated_chunks_gff=PATHS['chunks_gff'])
print(f'Gene MCDS dims: {dict(gene_mcds.dims)}')
print(f'DMW MCDS dims (curated):  {dict(dmw_mcds.dims)}')

## 2. Per-cluster methylation residuals

Aggregate mc/cov per (item × cluster), then apply logit-adjusted residuals
(per-cluster global offsets + per-item baseline). Run once for genes, once for DMWs.

In [ ]:
# Gene-level
g_c, g_m = du.aggregate_per_cluster(gene_mcds, meta, var_dim='gene')
g_obs, g_adj, g_filt, g_deltas = du.compute_logit_residuals(g_c, g_m)
print(f'Gene residuals (filtered): {g_filt.shape}')

In [ ]:
# DMW-level — restrict cells to col-0 for the canonical analysis
geno = du.derive_genotype_from_cellid(dmw_mcds['cell'].values)
col_cells = geno.index[geno == 'col']
d_c, d_m = du.aggregate_per_cluster(dmw_mcds, meta, var_dim='dmw',
                                    cell_subset=col_cells)
d_obs, d_adj, d_filt, d_deltas = du.compute_logit_residuals(d_c, d_m)
print(f'DMW residuals (filtered): {d_filt.shape}')

## 3. Gene-level WLS — Figure 1

WLS of zscored RNA on zscored methylation residual, weighted by coverage.
Then visualize the top-300 most anti-correlated genes side-by-side.

In [ ]:
common = g_filt.index.intersection(cluster_rna.index)
gene_stats = du.wls_correlate(g_filt.loc[common],
                              cluster_rna.loc[common],
                              g_m.loc[common])
gene_stats.to_csv(f'{DATA_DIR}/gene_wls_stats.tsv', sep='\t', index=False)
print(f'Tested genes: {len(gene_stats)}; FDR<0.05 negative: '
      f'{((gene_stats.fdr<0.05) & (gene_stats.rho<0)).sum()}')
gene_stats.sort_values('rho').head(10)

In [ ]:
TOP_N = 300
top_genes = gene_stats.sort_values('rho').head(TOP_N)['gene'].values
du.plot_paired_heatmap(
    g_filt.loc[top_genes], cluster_rna.loc[top_genes],
    title_meth='mCG gene-body residuals',
    outpath=f'{FIG_DIR}/fig1_gene_wls_top{TOP_N}.svg',
)
plt.show()

## 4. DMW-level WLS — Figure 2

Same idea, but DMWs are mapped to genes via gene-body overlap (`bioframe`),
and each DMW–gene pair is tested independently.

In [ ]:
dmw_to_gene = du.map_dmw_to_genes(dmw_mcds)
print(f'DMW–gene pairs: {len(dmw_to_gene)}  '
      f'({dmw_to_gene.dmw_id.nunique()} unique DMWs, '
      f'{dmw_to_gene.gene.nunique()} unique genes)')

In [ ]:
dmw_stats = du.wls_correlate(d_filt, cluster_rna, d_m, pairs=dmw_to_gene)
dmw_stats.to_csv(f'{DATA_DIR}/dmw_gene_wls_stats.tsv', sep='\t', index=False)
print(f'Tested pairs: {len(dmw_stats)}; FDR<0.05 negative: '
      f'{((dmw_stats.fdr<0.05) & (dmw_stats.rho<0)).sum()}')
dmw_stats.sort_values('rho').head(10)

In [ ]:
top_pairs = dmw_stats.sort_values('rho').head(TOP_N)
labels    = [f'{d} | {g}' for d, g in zip(top_pairs.dmw_id, top_pairs.gene)]
meth_rows = d_filt.loc[top_pairs.dmw_id.values].copy(); meth_rows.index = labels
rna_rows  = cluster_rna.loc[top_pairs.gene.values].copy(); rna_rows.index = labels
du.plot_paired_heatmap(
    meth_rows, rna_rows,
    title_meth='mCG DMW residuals',
    outpath=f'{FIG_DIR}/fig2_dmw_wls_top{TOP_N}.svg',
)
plt.show()

## 5. Cluster-specific hypomethylation — Table 1

Identify items whose methylation is residual-low AND z-low in specific cluster(s)
while being globally methylated. Run for genes and DMWs separately, then collapse
DMW hits back to gene level via the overlap map.

In [ ]:
gene_hypo = du.identify_relative_lows(g_filt.index, g_obs, g_adj, g_m)
gene_hypo.to_csv(f'{DATA_DIR}/hypomethylated_genes_by_cluster.csv', index=False)

dmw_hypo  = du.identify_relative_lows(d_filt.index, d_obs, d_adj, d_m)
dmw_hypo.to_csv(f'{DATA_DIR}/hypomethylated_dmw_events.csv', index=False)

dmw_hypo_genes = dmw_hypo.merge(dmw_to_gene, left_on='item', right_on='dmw_id')
dmw_hypo_genes.to_csv(f'{DATA_DIR}/hypomethylated_genes_via_dmw.csv', index=False)

print(f'Gene-level hypo events: {len(gene_hypo)} ({gene_hypo.item.nunique()} genes)')
print(f'DMW-level hypo events:  {len(dmw_hypo)}  ({dmw_hypo.item.nunique()} DMWs)')

## 6. col / rdd / met side-by-side — Figure 3

Per-genotype DMW residual matrices, deltas RE-FIT per genotype so each panel
shows within-genotype cluster structure. Rows anchored to the col-0 canonical
high-methylation set so the three panels align row-for-row.

In [ ]:
raw_by_geno, res_by_geno, canonical = du.compute_matrices_by_genotype(
    dmw_mcds, meta, canonical_rows_from='col')
for g in GENOTYPES:
    raw_by_geno[g].to_csv(f'{DATA_DIR}/{g}_dmw_raw_methylation.tsv', sep='\t')
    res_by_geno[g].to_csv(f'{DATA_DIR}/{g}_dmw_residual_methylation.tsv', sep='\t')
print(f'Canonical row set: {len(canonical)} DMWs')

In [ ]:
_, col_order = du.plot_genotype_panels(
    res_by_geno,
    outpath=f'{FIG_DIR}/fig3_col_rdd_met_residuals.svg',
    title='mCG DMW residuals — col / rdd / met (rows: col-0 Ward order)',
)
plt.show()

## 7. col-anchored bulk-shift — Figure 3 inset

Replace each genotype's null with col-0's `p0`. Now rdd panels shift red
(hypermethylated vs col) and met panels shift blue (hypomethylated vs col),
while cluster-specific structure remains visible on top.

Plotted un-z-scored on a shared ±0.7 diverging scale, using the col-0 row
order from §6 so panels line up row-for-row with Fig 3.

In [ ]:
anchored = du.compute_col_anchored_residuals(dmw_mcds, meta, canonical)
du.plot_genotype_panels(
    anchored, row_order=col_order, vmax=0.7,
    zscore_rows=False, show_bulk_shift=True,
    outpath=f'{FIG_DIR}/fig3b_col_rdd_met_anchored.svg',
    title='mCG DMW residuals — col-anchored (obs_p − p0(col))',
)
plt.show()

## 8. (Supplement) 9-condition subset heatmaps

Re-run the DMW WLS restricted to nine biological subsets of clusters
(All / Meta States / Mesophyll / Epidermis / Non-Meso-Epi / Terminal Non-Meso-Epi /
Terminal Ex-Guard / All Ex-Guard-G2M / Meso-and-Epi) and emit one heatmap each.
Subset definitions live in `du.make_cluster_subsets()`.

In [ ]:
# Re-label DMW/RNA matrix columns to human-readable cluster names once so the
# subset selection works in name-space (the helper expects column names).
d_filt_named = d_filt.rename(columns=CLUSTER_MAPPING)
d_m_named    = d_m.rename(columns=CLUSTER_MAPPING)
rna_named    = cluster_rna.rename(columns=CLUSTER_MAPPING)

subsets = du.make_cluster_subsets(d_filt_named.columns)
supp_summary = []
for slug, cols_sel in subsets.items():
    if len(cols_sel) < 3:
        continue
    stats = du.wls_correlate(d_filt_named[cols_sel],
                             rna_named[cols_sel],
                             d_m_named[cols_sel],
                             pairs=dmw_to_gene)
    if len(stats) == 0:
        continue
    top = stats.sort_values('rho').head(TOP_N)
    labels = [f'{d} | {g}' for d, g in zip(top.dmw_id, top.gene)]
    meth = d_filt_named.loc[top.dmw_id.values, cols_sel].copy(); meth.index = labels
    rna  = rna_named.loc[top.gene.values,  cols_sel].copy(); rna.index  = labels
    du.plot_paired_heatmap(
        meth, rna,
        title_meth=f'mCG DMW residuals — {slug}',
        ylabel=f'Top-{len(top)} DMW–gene pairs (per-subset WLS; rows differ across panels)',
        outpath=f'{FIG_DIR}/supp_dmw_wls_{slug}.svg',
    )
    plt.show()
    n_sig = ((stats.fdr < 0.05) & (stats.rho < 0)).sum()
    supp_summary.append((slug, len(cols_sel), len(stats), int(n_sig)))

pd.DataFrame(supp_summary, columns=['subset','n_clusters','n_tested','n_sig_neg'])